In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 1_presencas_eventos_ingest
# LOCAL: /1_bronze/src/
# OBJETIVO: Ingestão de dados de presença de deputados em eventos parlamentares
# ----------------------------------------------------------------------------------------


import requests as req
from pyspark.sql.functions import col

# Obter IDs dos eventos processados na Gold: evita puxar presenças de eventos que não irá usar
eventos_ids = spark.table("gold_fato_eventos").select("id_evento").distinct().collect()

lista_presencas_total = []

# Iteração sobre os eventos para buscar os deputados presentes
for row in eventos_ids:
    id_ev = row['id_evento']
    url = f"https://dadosabertos.camara.leg.br/api/v2/eventos/{id_ev}/deputados"
    
    try:
        res = req.get(url).json()
        for dep in res['dados']:            
            dep['id_evento'] = id_ev 
            lista_presencas_total.append(dep)
    except:
        print(f"Erro ao buscar evento {id_ev}")

# 3. Salvar na Bronze
if lista_presencas_total:
    df_presencas = spark.createDataFrame(lista_presencas_total)
    #df_presencas.write.mode("overwrite").saveAsTable("bronze_presenca_eventos")

    df_presencas.dropDuplicates(["id", "id_evento"]).write \
        .mode("overwrite") \
        .saveAsTable("bronze_presenca_eventos")

display(spark.table("bronze_presenca_eventos"))